# 02 — Exploratory Data Analysis

Explores the EDP SCADA and logbook data. Run `01_data_loading.ipynb` first to generate the processed parquets.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

from src.features import label_power_curve_status, compute_anomaly_windows, fleet_kpis
from src.rca import failure_pareto, downtime_pareto, monthly_failure_rate
from src.viz import plot_power_curve, plot_pareto, plot_temp_trend, plot_failure_heatmap

PROCESSED = repo_root / 'data' / 'processed'

In [ ]:
scada   = pd.read_parquet(PROCESSED / 'scada.parquet')
logbook = pd.read_parquet(PROCESSED / 'logbook.parquet')
print(f'SCADA: {scada.shape}  |  Logbook: {logbook.shape}')

## 1. Fleet KPIs

In [ ]:
kpis = fleet_kpis(scada, logbook)
for k, v in kpis.items():
    print(f'{k:30s}: {v}')

## 2. Power curve by turbine

In [ ]:
scada = label_power_curve_status(scada)

turbines = sorted(scada['turbine_id'].unique()) if 'turbine_id' in scada.columns else [None]
for tid in turbines[:3]:  # first 3 turbines
    fig = plot_power_curve(scada, turbine_id=tid)
    fig.show()

## 3. Wind speed distribution

In [ ]:
if 'wind_speed_ms' in scada.columns:
    fig = px.histogram(
        scada, x='wind_speed_ms', nbins=60,
        facet_col='turbine_id' if 'turbine_id' in scada.columns else None,
        facet_col_wrap=3,
        title='Wind Speed Distribution by Turbine',
        labels={'wind_speed_ms': 'Wind Speed (m/s)'},
    )
    fig.show()

## 4. Temperature correlation matrix

In [ ]:
temp_cols = [c for c in scada.columns if 'temp' in c]
if temp_cols:
    corr = scada[temp_cols].corr()
    fig = px.imshow(
        corr, text_auto='.2f',
        color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
        title='Temperature Column Correlations',
    )
    fig.show()
else:
    print('No temperature columns found.')

## 5. Operational status breakdown

In [ ]:
if 'operational_status' in scada.columns:
    counts = scada.groupby(['turbine_id','operational_status']).size().reset_index(name='count') \
             if 'turbine_id' in scada.columns else \
             scada['operational_status'].value_counts().reset_index()
    fig = px.bar(
        counts,
        x='turbine_id' if 'turbine_id' in counts.columns else 'operational_status',
        y='count',
        color='operational_status',
        barmode='stack',
        title='Operational Status Distribution per Turbine',
        color_discrete_map={
            'Normal':'#2ecc71','Underperforming':'#f39c12',
            'Fault/Stop':'#e74c3c','Curtailed':'#9b59b6','Low Wind':'#95a5a6'
        },
    )
    fig.show()

## 6. Failure Pareto

In [ ]:
pareto = failure_pareto(logbook)
display(pareto)
fig = plot_pareto(pareto)
fig.show()

In [ ]:
# Downtime Pareto
if 'duration_hours' in logbook.columns:
    dt_pareto = downtime_pareto(logbook)
    fig = plot_pareto(dt_pareto, value_label='Downtime (h)', title='Downtime Pareto by Component')
    fig.show()

## 7. Monthly failure heatmap

In [ ]:
monthly = monthly_failure_rate(logbook)
if not monthly.empty:
    fig = plot_failure_heatmap(monthly)
    fig.show()